# 07 - Preprocessing Pipeline (Exploratory Only)

**IMPORTANT:** This notebook is **exploratory only**.
---

**Input:** `data/processed/train_feat.csv` | `val_feat.csv` | `test_feat.csv`

**Output:** `artifacts/preprocessing_pipeline.pkl` (local only — not Git/DVC tracked; versioned via MLflow downstream)

**Config:** `configs/data_config.yaml` → `features_config` section

### What this notebook does (Exploratory)

| Step | Action | Source / Note |
| :--- | :--- | :--- |
| 1 | Setup environment, imports, and `PROJECT_DIR` | `src.utils.paths` |
| 2 | Load feature-engineered splits from `data/processed/` | `DataLoader` + empty checks |
| 3 | Verify column assignment to transformers | `resolve_columns()` |
| 4 | Run preprocessing pipeline (fit on train only) | `run_pipeline()` with `save=True`, `auto_track_dvc=False` |
| 5 | Verify results (shapes, NaN, Inf, feature names) | Exploratory validation only |
| 6 | Check scaling sanity (mean~0, std~1 for StandardScaler) | Exploratory validation only |
| 7 | Check OHE expansion (ISLAND handling) | Exploratory validation only |
| 8 | Check target distribution (not transformed here) | Confirm target is untouched |
| 9 | Verify artifact saved | `artifacts/preprocessing_pipeline.pkl` |

**Note:** No DVC or Git operations are performed in this notebook.

### FIT/TRANSFORM rule

Pipeline is **fit on train only** and applied to val/test using train-derived statistics.
This prevents data leakage from val/test distribution into model training.

---
## 0 - Setup & Load

In [ ]:
# ===================================================================
# Section 0 · Setup
# ===================================================================
import os
import sys
from pathlib import Path

from src.utils.paths import PROJECT_DIR

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
importlib.invalidate_caches()

print(f"✅ Working dir : {os.getcwd()}")
print(f"✅ sys.path[0] : {sys.path[0]}")

In [ ]:
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.utils.logger import setup_logging, get_logger
from src.data.data_loader import DataLoader
from src.features.pipeline import (
    run_pipeline,
    load_pipeline,
    resolve_columns,
    get_feature_names,
    PipelineResult,
    _STD_SCALE_COLS,
    _ROBUST_SCALE_COLS,
    _CAT_COLS,
    _PASSTHROUGH_COLS,
    _TARGET,
)

setup_logging(level=logging.INFO)
logger = get_logger("notebook.06_preprocessing")

print("Imports ready")

---
## 2 - Load Feature-Engineered Splits

In [ ]:
loader = DataLoader()

train = loader.load_processed("train_feat.csv")
val = loader.load_processed("val_feat.csv")
test = loader.load_processed("test_feat.csv")

print(f"train : {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"val   : {val.shape[0]:,} rows x {val.shape[1]} cols")
print(f"test  : {test.shape[0]:,} rows x {test.shape[1]} cols")
print()
print("Columns:")
print(train.columns.tolist())

---
## 3 - Column Assignment Check

Verify each column is routed to the correct transformer before fitting.

In [ ]:
cols = resolve_columns(train)

print("-- Column assignment --")
print(f"  StandardScaler ({len(cols['std'])} cols)      : {cols['std']}")
print(f"  RobustScaler   ({len(cols['robust'])} col)     : {cols['robust']}")
print(f"  OneHotEncoder  ({len(cols['cat'])} col)     : {cols['cat']}")
print(f"  Passthrough    ({len(cols['passthrough'])} cols)      : {cols['passthrough']}")
print()

# Verify target is not in any group
all_assigned = cols['std'] + cols['robust'] + cols['cat'] + cols['passthrough']
assert _TARGET not in all_assigned, f"Target '{_TARGET}' found in transformer groups!"
print(f"Target '{_TARGET}' correctly excluded from all transformer groups")

---
## 4 - Run Preprocessing Pipeline

`run_pipeline()` does the following in order:
1. Resolve columns to transformer groups
2. Build `ColumnTransformer` with `StandardScaler`, `RobustScaler`, `OneHotEncoder`
3. **Fit on train only** (no leakage)
4. Transform all three splits
5. Save fitted pipeline to `artifacts/preprocessing_pipeline.pkl` (local, untracked)

In [ ]:
import os
from pathlib import Path

# Ensure artifacts directory exists (local only — not Git/DVC tracked)
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

result = run_pipeline(
    train=train,
    val=val,
    test=test,
    artifacts_dir=artifacts_dir,   # <- save pkl here (local only)
    auto_track_dvc=False,          # <- no DVC
    save=True,
)

print(result.summary())

---
## 5 - Verify Preprocessing Results

Four checks:
- **Shapes** — correct row/column counts
- **No NaN** — no missing values in output arrays
- **No Inf** — no infinite values
- **Feature names** — OHE expanded correctly

In [ ]:
print("-- Shape check --")
for name, arr in [("X_train", result.X_train), ("X_val", result.X_val), ("X_test", result.X_test)]:
    print(f"  {name:<10} : {arr.shape}")
print(f"  n_features : {result.n_features}")
print()
assert result.X_train.shape[1] == result.X_val.shape[1] == result.X_test.shape[1]
print("All splits have same number of features - OK")

In [ ]:
print("-- NaN check --")
for name, arr in [("X_train", result.X_train), ("X_val", result.X_val), ("X_test", result.X_test)]:
    n_nan = np.isnan(arr).sum()
    status = "OK" if n_nan == 0 else f"FAIL ({n_nan} NaNs)"
    print(f"  {name:<10} : {status}")

print()
print("-- Inf check --")
for name, arr in [("X_train", result.X_train), ("X_val", result.X_val), ("X_test", result.X_test)]:
    n_inf = np.isinf(arr).sum()
    status = "OK" if n_inf == 0 else f"FAIL ({n_inf} Infs)"
    print(f"  {name:<10} : {status}")

In [ ]:
print("-- Feature names --")
for i, name in enumerate(result.feature_names_out):
    tag = "[OHE]" if "ocean_proximity" in name else "[NUM]" if name not in _PASSTHROUGH_COLS else "[PASS]"
    print(f"  {i:>3}  {tag}  {name}")

print()
ohe_cols = [n for n in result.feature_names_out if "ocean_proximity" in n]
print(f"OHE expanded ocean_proximity into {len(ohe_cols)} columns: {ohe_cols}")

---
## 6 - Scaling Sanity Check

Verify that StandardScaler and RobustScaler applied correctly:
- StandardScaler output should have mean ~0 and std ~1 on train
- RobustScaler output should be centred around 0

In [ ]:
df_out = pd.DataFrame(result.X_train, columns=result.feature_names_out)

print("-- StandardScaler columns (train) — expect mean~0, std~1 --")
std_present = [c for c in _STD_SCALE_COLS if c in result.feature_names_out]
if std_present:
    stats = df_out[std_present].agg(["mean", "std"]).T.round(4)
    print(stats.to_string())

print()
print("-- RobustScaler columns (train) — expect centred around 0 --")
robust_present = [c for c in _ROBUST_SCALE_COLS if c in result.feature_names_out]
if robust_present:
    stats = df_out[robust_present].agg(["mean", "std"]).T.round(4)
    print(stats.to_string())

print()
print("-- Passthrough columns (should be unchanged) --")
pass_present = [c for c in _PASSTHROUGH_COLS if c in result.feature_names_out]
if pass_present:
    for col in pass_present:
        unique_vals = sorted(df_out[col].unique().tolist())
        print(f"  {col}: unique values = {unique_vals} (expected: 0 and/or 1)")

---
## 7 - Feature Distribution After Scaling

In [ ]:
BG = "#0d1117"
AX_BG = "#161b22"
TEXT = "#e6edf3"
MUTED = "#8b949e"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": AX_BG,
    "axes.edgecolor": "#30363d", "axes.labelcolor": TEXT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": TEXT, "grid.color": "#30363d",
    "grid.alpha": 0.5, "font.family": "monospace",
})

num_cols_to_plot = [c for c in result.feature_names_out
                    if c not in _PASSTHROUGH_COLS
                    and "ocean_proximity" not in c]

n_cols = 3
n_rows = int(np.ceil(len(num_cols_to_plot) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows), facecolor=BG)
axes = axes.flatten()

for i, col in enumerate(num_cols_to_plot):
    ax = axes[i]
    ax.hist(df_out[col], bins=40, color="#58a6ff", edgecolor=BG, linewidth=0.2, alpha=0.85)
    mean_val = df_out[col].mean()
    ax.axvline(mean_val, color="#f85149", lw=1.5, ls="--")
    ax.set_title(f"{col}\nmean={mean_val:.2f}", color=TEXT, fontsize=9)
    ax.grid(axis="y", alpha=0.3)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature Distributions After Preprocessing (train)", fontsize=13, color=TEXT, y=1.01)
plt.tight_layout()
plt.savefig("preprocessing_distributions.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()

---
## 8 - OHE Category Check

Verify `ocean_proximity` expanded correctly and ISLAND is handled.

In [ ]:
ohe_cols = [c for c in result.feature_names_out if "ocean_proximity" in c]

print("-- OHE categories in train --")
for col in ohe_cols:
    unique_vals = sorted(df_out[col].unique().tolist())
    count_ones = int(df_out[col].sum())
    print(f"  {col:<35} unique={unique_vals}  count_1={count_ones}")

print()
print("-- ISLAND handling check (should be all zeros if not in train) --")
island_col = [c for c in ohe_cols if "ISLAND" in c.upper()]
if island_col:
    island_sum = df_out[island_col[0]].sum()
    print(f"  {island_col[0]}: sum={island_sum} ({'seen in train' if island_sum > 0 else 'not in train - OHE zeros as expected'})")
else:
    print("  ISLAND not in train -> column not created (handle_unknown=ignore covers it in val/test)")

---
## 9 - Target Distribution Check

The target `median_house_value` was NOT transformed by the pipeline.
log1p transform on the target is applied inside the model training step.

In [ ]:
print("-- Target (y) statistics --")
for name, y in [("y_train", result.y_train), ("y_val", result.y_val), ("y_test", result.y_test)]:
    print(f"  {name:<10} : min=${y.min():,.0f}  max=${y.max():,.0f}  mean=${y.mean():,.0f}  nulls={y.isnull().sum()}")

print()
print("Note: log1p transform on target will be applied in train.py (not here)")

---
## 10 - Artifact Check

In [ ]:
from pathlib import Path

artifacts_dir = Path("artifacts")
pkl_path = artifacts_dir / "preprocessing_pipeline.pkl"

print("-- Artifact file --")
if pkl_path.exists():
    size_kb = pkl_path.stat().st_size / 1024
    print(f"  OK  {pkl_path}  ({size_kb:.1f} KB)")
    print(f"  -> Full path: {pkl_path.resolve()}")
else:
    print(f"  MISSING  {pkl_path}")
    print("  Run Section 5 first.")

print()
print("-- Round-trip check (load and re-transform train) --")
if pkl_path.exists():
    loaded_pipeline = load_pipeline(artifacts_dir=artifacts_dir)
    X_reloaded = loaded_pipeline.transform(train.drop(columns=[_TARGET]))
    match = np.allclose(result.X_train, X_reloaded)
    print(f"  Loaded pipeline produces identical output: {'OK' if match else 'MISMATCH'}")


## Summary & Next Steps

| Done | Details |
| :--- | :--- |
| StandardScaler | `longitude`, `latitude`, `housing_median_age`, ratios, distances |
| RobustScaler | `median_income` (skew=1.626, not a count) |
| OneHotEncoder | `ocean_proximity` — handle_unknown=ignore for ISLAND |
| Passthrough | `lof_outlier` (binary flag) |
| Pipeline saved | `artifacts/preprocessing_pipeline.pkl` (local only, untracked — versioned via MLflow) |

